# Momentum Trading & Deep Alpha Pillars Analysis

This notebook performs a momentum trading analysis on a selected list of stocks and layers on the Deep Alpha Copilot "Pillars" evaluation framework to generate Buy/Hold/Sell recommendations.

**Goal:** Identify stocks with strong momentum and solid fundamentals based on the 7 Deep Alpha Pillars, combined with Macroeconomic context.

**Methodology:**
1. **Data Collection**: Fetch technical data (Returns, MA, RSI, Volume) and Macro data (VIX, Yields, Sector Performance).
2. **Deep Alpha Scoring**: Use the internal scoring engine to evaluate stocks across 7 business pillars.
3. **Scoring & Recommendation**: Apply specific filters (Trend Confirmation, Relative Strength, Demand, Macro) to generate actionable advice.

In [1]:
# !pip install -r deep_alpha_copilot/requirements.txt
# !pip install yfinance pandas numpy vaderSentiment matplotlib

In [2]:
import sys
import os
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime, timedelta

# Add the project root to sys.path to allow imports from the app
project_root = os.path.abspath(os.path.join(os.getcwd(), 'deep_alpha_copilot'))
if project_root not in sys.path:
    sys.path.append(project_root)

try:
    from app.scoring.engine import compute_company_scores
except ImportError as e:
    print(f"Warning: Could not import local modules ({e}).")

# Updated Ticker List from User Instructions (Total 27)
TARGET_TICKERS = [
    'INTC', 'LAC', 'MU', 'CCJ', 'URNM', 'NLR', 'NUKZ', 'URA', 
    'NVDA', 'MSFT', 'LEU', 'IREN', 'CLS', 'ANET', 'OKLO', 'QBTS', 'NAK', 
    'SMAR', 'UNH', 'RKLB', 'FLNC', 'ASST', 'EOSE', 'BE', 'SNDK', 'CIFR',
    'BE', 'INTC'
    
]

print(f"Analyzing {len(TARGET_TICKERS)} Tickers: {TARGET_TICKERS}")

Analyzing 28 Tickers: ['INTC', 'LAC', 'MU', 'CCJ', 'URNM', 'NLR', 'NUKZ', 'URA', 'NVDA', 'MSFT', 'LEU', 'IREN', 'CLS', 'ANET', 'OKLO', 'QBTS', 'NAK', 'SMAR', 'UNH', 'RKLB', 'FLNC', 'ASST', 'EOSE', 'BE', 'SNDK', 'CIFR', 'BE', 'INTC']


## 1. Macroeconomic & Market Context

We assess the macro environment to see if it supports momentum strategies.

**Key Metrics:**
- **Index Performance (SPY/QQQ)**: Check if market is bullish.
- **VIX**: Volatility Index (Low/Falling is good).
- **10-Year Yield (^TNX)**: Interest rate trends.
- **Sector Performance**: Identify leading sectors.

In [3]:
def fetch_macro_data():
    tickers = ['SPY', 'QQQ', '^VIX', '^TNX']
    data = yf.download(tickers, period="1y", progress=False)['Close']
    
    macro_summary = {}
    
    # 1. Index Performance (YTD approx from start of year, here simplified to 1y trend)
    # YTD Return Calculation
    start_of_year = datetime(datetime.now().year, 1, 1)
    ytd_start = data.index[data.index >= start_of_year][0]
    
    for idx in ['SPY', 'QQQ']: 
        if idx in data.columns:
            current = data[idx].iloc[-1]
            start = data[idx].loc[ytd_start]
            ytd_ret = (current / start) - 1
            macro_summary[f'{idx}_YTD'] = ytd_ret
    
    # 2. VIX Level
    if '^VIX' in data.columns:
        macro_summary['VIX'] = data['^VIX'].iloc[-1]
    
    # 3. 10-Year Yield
    if '^TNX' in data.columns:
        macro_summary['10Y_Yield'] = data['^TNX'].iloc[-1]
    
    return macro_summary

try:
    macro_data = fetch_macro_data()
    print("Macro Data Summary:")
    for k, v in macro_data.items():
        print(f"{k}: {v:.2f}" if 'YTD' not in k else f"{k}: {v:.2%}")
except Exception as e:
    print(f"Error fetching macro data: {e}")
    macro_data = {}

/var/folders/p8/7ctr71895j53jvb8h5hz83rm0000gn/T/ipykernel_43835/2233722779.py:3: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers, period="1y", progress=False)['Close']


Macro Data Summary:
SPY_YTD: 15.39%
QQQ_YTD: 19.05%
VIX: 20.52
10Y_Yield: 4.04


## 2. Individual Stock Momentum Data

We calculate specific technical indicators for each of the 27 stocks:
- **Price & Returns**: Current, 6M %, 12M %
- **Trend**: 50D MA, 200D MA
- **Strength**: RSI (14)
- **Volume**: ADV (30D), Relative Volume (RVOL)

In [4]:
def compute_rsi(series, period=14):
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

def fetch_momentum_data(tickers, period="2y"):
    print("Fetching stock data... (This may take a moment)")
    # Fetch Close and Volume
    data = yf.download(tickers, period=period, group_by='ticker', progress=False)
    
    results = []
    
    for ticker in tickers:
        try:
            # Extract data for single ticker
            if isinstance(data.columns, pd.MultiIndex):
                if ticker not in data.columns.levels[0]:
                    print(f"Data missing for {ticker}")
                    continue
                df = data[ticker].copy()
            else:
                df = data.copy()
            
            if df.empty:
                continue
                
            # Clean data
            closes = df['Close'].fillna(method='ffill')
            volume = df['Volume'].fillna(0)
            
            # Metrics Calculation
            current_price = closes.iloc[-1]
            
            # 6 Month Return (126 days)
            r6m = (current_price / closes.iloc[-126] - 1) if len(closes) > 126 else np.nan
            
            # 12 Month Return (252 days)
            r12m = (current_price / closes.iloc[-252] - 1) if len(closes) > 252 else np.nan
            
            # Moving Averages
            ma50 = closes.rolling(window=50).mean().iloc[-1]
            ma200 = closes.rolling(window=200).mean().iloc[-1]
            
            # RSI
            rsi_series = compute_rsi(closes)
            rsi = rsi_series.iloc[-1]
            
            # Volume Metrics
            adv30 = volume.rolling(window=30).mean().iloc[-1]
            current_vol = volume.iloc[-1]
            rvol = current_vol / adv30 if adv30 > 0 else np.nan
            
            results.append({
                'Ticker': ticker,
                'Price': current_price,
                'R6M': r6m,
                'R12M': r12m,
                'MA50': ma50,
                'MA200': ma200,
                'RSI': rsi,
                'ADV30': adv30,
                'RVOL': rvol
            })
            
        except Exception as e:
            print(f"Error processing {ticker}: {e}")
            
    return pd.DataFrame(results).set_index('Ticker')

momentum_df = fetch_momentum_data(TARGET_TICKERS)

if not momentum_df.empty:
    print("\nMomentum Data (Top 5 by 6M Return):")
    display(momentum_df.sort_values('R6M', ascending=False).head())

Fetching stock data... (This may take a moment)


/var/folders/p8/7ctr71895j53jvb8h5hz83rm0000gn/T/ipykernel_43835/812129126.py:11: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers, period=period, group_by='ticker', progress=False)

1 Failed download:
['SMAR']: YFPricesMissingError('possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")')



Momentum Data (Top 5 by 6M Return):


/var/folders/p8/7ctr71895j53jvb8h5hz83rm0000gn/T/ipykernel_43835/812129126.py:30: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  closes = df['Close'].fillna(method='ffill')
/var/folders/p8/7ctr71895j53jvb8h5hz83rm0000gn/T/ipykernel_43835/812129126.py:30: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  closes = df['Close'].fillna(method='ffill')
/var/folders/p8/7ctr71895j53jvb8h5hz83rm0000gn/T/ipykernel_43835/812129126.py:30: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  closes = df['Close'].fillna(method='ffill')
/var/folders/p8/7ctr71895j53jvb8h5hz83rm0000gn/T/ipykernel_43835/812129126.py:30: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  

,Price,R6M,R12M,MA50,MA200,RSI,ADV30,RVOL
Ticker,,,,,,,,
SNDK,226.960007,4.882841,NaN,166.246801,NaN,55.976676,1.203141e+07,1.112637
IREN,48.490002,4.472912,3.988683,54.191500,23.062575,33.545001,4.174702e+07,0.871394
CIFR,16.709999,3.988060,1.699515,16.786000,7.621350,31.714999,5.133491e+07,0.856528
BE,95.559998,3.877999,2.991646,102.941200,46.499700,33.581717,1.589075e+07,1.745236
BE,95.559998,3.877999,2.991646,102.941200,46.499700,33.581717,1.589075e+07,1.745236


## 3. Deep Alpha Pillars Analysis

We compute fundamental scores using the `deep_alpha_copilot` engine. 
**Strict Check**: We filter out default/placeholder scores if underlying data is missing.

In [5]:
pillar_scores = []

print("Calculating Pillar Scores (this checks local data)...")
for ticker in TARGET_TICKERS:
    try:
        # Note: If local data files (JSON/CSV) are missing for new tickers, this might return empty/default scores.
        scores = compute_company_scores(ticker)
        
        if scores and scores.get('scores'):
            overall = scores.get('overall', {})
            comp_scores = scores.get('scores', {})
            
            # VALIDATION: Check if the score is based on real data
            # If sector is unknown AND total inputs are near zero, it's likely a default placeholder (5.0)
            sector = scores.get('company', {}).get('sector')
            # Count how many real inputs went into the score components
            total_inputs = sum(len(c.get('inputs', {})) for c in comp_scores.values() if isinstance(c, dict))
            
            if not sector and total_inputs == 0:
                da_score = np.nan
                summary = "Insufficient Fundamental Data"
            else:
                da_score = overall.get('score', 0)
                summary = scores.get('recommendation', {}).get('why_buy') or "N/A"

            row = {
                'Ticker': ticker,
                'DeepAlpha_Score': da_score,
                'Sector': sector or 'Unknown',
                'Industry': scores.get('company', {}).get('industry', 'Unknown'),
                'Summary': summary
            }
            pillar_scores.append(row)
            
    except Exception as e:
        # print(f"Error scoring {ticker}: {e}") # Suppress noise for missing data
        pass

pillars_df = pd.DataFrame(pillar_scores)
if not pillars_df.empty:
    pillars_df = pillars_df.set_index('Ticker')
    print("Pillar Data Loaded.")

Calculating Pillar Scores (this checks local data)...
Pillar Data Loaded.


## 4. Final Scoring & Recommendation Logic

**Criteria:**
1. **Fundamental Health**: Deep Alpha Score > 5.0 (implied sound "why").
2. **Trend Confirmation**: Price > 50MA > 200MA.
3. **Relative Strength**: Strong R6M and R12M.
4. **Demand**: RVOL > 1.0 and RSI 50-70 (bullish but not overbought).

In [6]:
if not momentum_df.empty:
    # Merge Pillar data if available, else treat as missing
    if not pillars_df.empty:
        final_df = momentum_df.join(pillars_df, how='left')
    else:
        final_df = momentum_df.copy()
        final_df['DeepAlpha_Score'] = np.nan
        final_df['Sector'] = 'Unknown'

    recommendations = []
    
    for ticker, row in final_df.iterrows():
        price = row['Price']
        ma50 = row['MA50']
        ma200 = row['MA200']
        rsi = row['RSI']
        r6m = row['R6M']
        r12m = row['R12M']
        da_score = row.get('DeepAlpha_Score', np.nan)
        
        score_log = []
        points = 0
        
        # 1. Trend Check
        if price > ma50 and ma50 > ma200:
            points += 2
            score_log.append("Strong Uptrend (Price > 50 > 200)")
        elif price > ma50:
            points += 1
            score_log.append("Above 50MA")
            
        # 2. Relative Strength
        if r6m > 0.20: # >20% in 6m
            points += 1
            score_log.append("Strong 6M Mom")
        if r12m > 0.30: # >30% in 12m
            points += 1
            score_log.append("Strong 12M Mom")
            
        # 3. Demand / RSI
        if 50 <= rsi <= 70:
            points += 1
            score_log.append("Ideal RSI (50-70)")
        elif rsi > 70:
            score_log.append("Overbought (RSI > 70)")
            
        # 4. Fundamental (Deep Alpha)
        if pd.isna(da_score):
            score_log.append("Fundamentals Missing")
        elif da_score >= 6.5:
            points += 2
            score_log.append("Strong Fundamentals")
        elif da_score < 4.0:
            points -= 2
            score_log.append("Weak Fundamentals")
            
        # Determine Action
        if points >= 5:
            action = "BUY"
        elif points >= 3:
            action = "HOLD"
        else:
            action = "SELL"
            
        recommendations.append({
            'Ticker': ticker,
            'Action': action,
            'Price': f"${price:.2f}",
            'RSI': f"{rsi:.1f}",
            'R6M': f"{r6m:.1%}" if pd.notnull(r6m) else "N/A",
            'Trend': "Bullish" if price > ma50 > ma200 else "Mixed/Bearish",
            'DeepAlpha_Score': da_score if pd.notnull(da_score) else "N/A",
            'Reasoning': "; ".join(score_log)
        })
        
    final_results = pd.DataFrame(recommendations).set_index('Ticker')
    
    # Sort by Action strength (Buy -> Hold -> Sell) and then DeepAlpha Score
    action_order = {"BUY": 0, "HOLD": 1, "SELL": 2}
    final_results['Action_Rank'] = final_results['Action'].map(action_order)
    # Handle sorting with strings/NaN mixed in DeepAlpha_Score for display logic
    final_results['Sort_Score'] = pd.to_numeric(final_results['DeepAlpha_Score'], errors='coerce').fillna(0)
    final_results = final_results.sort_values(['Action_Rank', 'Sort_Score'], ascending=[True, False])
    final_results = final_results.drop(['Action_Rank', 'Sort_Score'], axis=1)
    
    print("\n=== FINAL MOMENTUM PORTFOLIO RECOMMENDATIONS ===")
    display(final_results)
    
    # final_results.to_csv('momentum_portfolio_results.csv')
else:
    print("No data to process.")


=== FINAL MOMENTUM PORTFOLIO RECOMMENDATIONS ===


,Action,Price,RSI,R6M,Trend,DeepAlpha_Score,Reasoning
Ticker,,,,,,,
MU,BUY,$223.93,52.1,133.2%,Bullish,7.77,Strong Uptrend (Price > 50 > 200); Strong 6M M...
CLS,BUY,$322.54,47.3,173.1%,Bullish,6.68,Strong Uptrend (Price > 50 > 200); Strong 6M M...
NVDA,HOLD,$182.55,37.3,35.4%,Mixed/Bearish,7.88,Strong 6M Mom; Strong Fundamentals
IREN,HOLD,$48.49,33.5,447.3%,Mixed/Bearish,7.67,Strong 6M Mom; Strong 12M Mom; Strong Fundamen...
ANET,HOLD,$122.17,20.0,31.7%,Mixed/Bearish,7.19,Strong 6M Mom; Strong Fundamentals
OKLO,HOLD,$89.55,36.6,62.1%,Mixed/Bearish,6.94,Strong 6M Mom; Strong 12M Mom; Strong Fundamen...
NAK,HOLD,$1.56,33.8,43.1%,Mixed/Bearish,6.72,Strong 6M Mom; Strong 12M Mom; Strong Fundamen...
LAC,HOLD,$4.87,55.1,75.2%,Mixed/Bearish,6.67,Strong 6M Mom; Ideal RSI (50-70); Strong Funda...
INTC,HOLD,$35.79,45.0,75.7%,Mixed/Bearish,6.65,Strong 6M Mom; Strong 12M Mom; Strong Fundamen...


## 5. Historical Grid Search Backtest

We perform a historical "Walk-Forward" simulation to test different "Buy the Dip" rule combinations over the past 2 years.

**Trading Strategy Under Test:**
1. **Trigger**: Stock drops X% in N days.
2. **Filter**: Must meet Trend/RSI/Fundamental criteria.
3. **Exit**: Hold for H days.

**Strategies Tested:**
- **Minervini Trend**: `SMA50 > SMA200` (Only buy dips in established uptrends).
- **O'Neil RS**: `Stock 12M > SPY 12M` (Only buy leaders).
- **Mean Reversion**: `RSI < 40` (Aggressive) vs `RSI < 30` (Conservative).
- **Deep Alpha Filter**: `Score > 6.0` (Fundamentals must be solid).

In [7]:
def fetch_historical_data(tickers, benchmark='SPY', period="2y"):
    print("Fetching historical OHLCV for backtest... (2 Years)")
    # Download all at once
    all_tickers = list(set(tickers + [benchmark]))
    data = yf.download(all_tickers, period=period, group_by='ticker', progress=False)
    return data

def backtest_strategy(data, tickers, benchmark_ticker, params, pillar_scores_map):
    """
    Runs a single parameter combination backtest over the history.
    """
    trades = []
    
    # Unpack params
    dip_days = params['dip_days']       # e.g., 10 days to measure drop
    dip_thresh = params['dip_thresh']   # e.g., 0.10 (10% drop)
    trend_mode = params['trend_mode']   # 'None', 'SMA200', 'GoldenCross'
    rsi_thresh = params['rsi_thresh']   # 'None', 40, 30
    fund_filter = params['fund_filter'] # True/False (Use Deep Alpha Score > 6.0)
    hold_days = params['hold_days']     # e.g., 10 days
    
    # Get Benchmark Data for RS Filter
    if benchmark_ticker in data.columns.levels[0]:
        spy_close = data[benchmark_ticker]['Close']
    else:
        spy_close = pd.Series(dtype=float)

    for ticker in tickers:
        if ticker == benchmark_ticker: continue
        if ticker not in data.columns.levels[0]: continue
        
        df = data[ticker].copy()
        if df.empty: continue
        
        # Pre-calculate Indicators
        close = df['Close'].fillna(method='ffill')
        ma50 = close.rolling(50).mean()
        ma200 = close.rolling(200).mean()
        
        # RSI
        delta = close.diff()
        gain = (delta.where(delta > 0, 0)).rolling(14).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
        rs = gain / loss
        rsi = 100 - (100 / (1 + rs))
        
        # Dip Calculation (Price vs Price N days ago)
        # We want: (Current / Lagged) - 1 <= -dip_thresh
        price_lagged = close.shift(dip_days)
        drop_pct = (close / price_lagged) - 1
        
        # Fundamental Check (Static for now)
        da_score = pillar_scores_map.get(ticker, 0)
        pass_fund = True
        if fund_filter and (pd.isna(da_score) or da_score < 6.0):
            pass_fund = False
            
        # Iterate through days (start after MA200 warmup)
        # Vectorized approach is faster, but loop is clearer for logic
        # We'll loop for clarity in this demo
        start_idx = 200
        for i in range(start_idx, len(df) - hold_days):
            date = df.index[i]
            curr_price = close.iloc[i]
            
            # 1. Check Dip
            if drop_pct.iloc[i] > -dip_thresh:
                continue # Not a dip
            
            # 2. Fundamental Filter
            if not pass_fund:
                continue
                
            # 3. Technical Filters
            # Trend
            pass_trend = True
            if trend_mode == 'SMA200':
                if curr_price < ma200.iloc[i]: pass_trend = False
            elif trend_mode == 'GoldenCross':
                if not (ma50.iloc[i] > ma200.iloc[i]): pass_trend = False
            if not pass_trend: continue
            
            # RSI
            if rsi_thresh != 'None':
                if rsi.iloc[i] > rsi_thresh: continue
            
            # EXECUTE TRADE
            # Entry @ Close[i]
            # Exit @ Close[i + hold_days]
            exit_price = close.iloc[i + hold_days]
            trade_ret = (exit_price / curr_price) - 1
            
            trades.append({
                'Ticker': ticker,
                'Date': date,
                'Return': trade_ret,
                'Strategy': params['name']
            })
            
    return pd.DataFrame(trades)

In [8]:
# 1. Get Data
hist_data = fetch_historical_data(TARGET_TICKERS)

# 2. Map Scores
pillar_map = {}
if not pillars_df.empty:
    pillar_map = pillars_df['DeepAlpha_Score'].to_dict()

# 3. Grid Settings
grid_configs = [
    # BASELINE: Simple Dip Buying
    {'name': 'Dip 10%', 'dip_days': 5, 'dip_thresh': 0.10, 'trend_mode': 'None', 'rsi_thresh': 'None', 'fund_filter': False, 'hold_days': 10},
    {'name': 'Dip 15%', 'dip_days': 10, 'dip_thresh': 0.15, 'trend_mode': 'None', 'rsi_thresh': 'None', 'fund_filter': False, 'hold_days': 10},
    
    # TREND: Minervini Style
    {'name': 'Minervini (GoldenCross)', 'dip_days': 5, 'dip_thresh': 0.10, 'trend_mode': 'GoldenCross', 'rsi_thresh': 'None', 'fund_filter': False, 'hold_days': 10},
    
    # RSI: Mean Reversion
    {'name': 'RSI < 40 (Aggressive)', 'dip_days': 5, 'dip_thresh': 0.05, 'trend_mode': 'SMA200', 'rsi_thresh': 40, 'fund_filter': False, 'hold_days': 5},
    {'name': 'RSI < 30 (Panic)', 'dip_days': 5, 'dip_thresh': 0.05, 'trend_mode': 'None', 'rsi_thresh': 30, 'fund_filter': False, 'hold_days': 5},
    
    # FUNDAMENTAL: Deep Alpha Boost
    {'name': 'Deep Alpha Only (>6.0)', 'dip_days': 5, 'dip_thresh': 0.10, 'trend_mode': 'None', 'rsi_thresh': 'None', 'fund_filter': True, 'hold_days': 10},
    {'name': 'Alpha + Minervini', 'dip_days': 5, 'dip_thresh': 0.10, 'trend_mode': 'GoldenCross', 'rsi_thresh': 'None', 'fund_filter': True, 'hold_days': 20}
]

results_summary = []
all_trades_list = []

print("Running Grid Search...\n")

# Configure pandas to show more rows
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

for cfg in grid_configs:
    res_df = backtest_strategy(hist_data, TARGET_TICKERS, 'SPY', cfg, pillar_map)
    
    if not res_df.empty:
        avg_ret = res_df['Return'].mean()
        win_rate = (res_df['Return'] > 0).mean()
        count = len(res_df)
        all_trades_list.append(res_df)
    else:
        avg_ret = 0
        win_rate = 0
        count = 0
        
    results_summary.append({
        'Strategy': cfg['name'],
        'Win Rate': f"{win_rate:.1%}",
        'Avg Return': f"{avg_ret:.2%}",
        '# Trades': count,
        'Hold Days': cfg['hold_days']
    })

# Display Overall Leaderboard
leaderboard = pd.DataFrame(results_summary).sort_values('Avg Return', ascending=False)
print("=== Overall Strategy Leaderboard ===")
display(leaderboard)

# --- Detailed Breakdown: Per Stock Per Strategy ---
if all_trades_list:
    all_trades = pd.concat(all_trades_list)
    
    # Group by Strategy AND Ticker
    grouped = all_trades.groupby(['Strategy', 'Ticker'])['Return'].agg(['count', 'mean', lambda x: (x > 0).mean()])
    grouped.columns = ['# Trades', 'Avg Return', 'Win Rate']
    
    # Format for display
    grouped['Avg Return'] = grouped['Avg Return'].map(lambda x: f"{x:.2%}")
    grouped['Win Rate'] = grouped['Win Rate'].map(lambda x: f"{x:.1%}")
    
    print("\n=== Detailed Breakdown by Stock (All Strategies) ===")
    display(grouped)
else:
    print("No trades found.")

Fetching historical OHLCV for backtest... (2 Years)


/var/folders/p8/7ctr71895j53jvb8h5hz83rm0000gn/T/ipykernel_43835/479243729.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(all_tickers, period=period, group_by='ticker', progress=False)

1 Failed download:
['SMAR']: YFPricesMissingError('possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")')
/var/folders/p8/7ctr71895j53jvb8h5hz83rm0000gn/T/ipykernel_43835/479243729.py:36: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  close = df['Close'].fillna(method='ffill')
/var/folders/p8/7ctr71895j53jvb8h5hz83rm0000gn/T/ipykernel_43835/479243729.py:36: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  close = df['Close'].fillna(method='ffill')
/var/folders/p8/7ctr71895j53jvb8h5hz83rm0000gn/T/ipykernel_43835/479243729.

Running Grid Search...



/var/folders/p8/7ctr71895j53jvb8h5hz83rm0000gn/T/ipykernel_43835/479243729.py:36: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  close = df['Close'].fillna(method='ffill')
/var/folders/p8/7ctr71895j53jvb8h5hz83rm0000gn/T/ipykernel_43835/479243729.py:36: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  close = df['Close'].fillna(method='ffill')
/var/folders/p8/7ctr71895j53jvb8h5hz83rm0000gn/T/ipykernel_43835/479243729.py:36: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  close = df['Close'].fillna(method='ffill')
/var/folders/p8/7ctr71895j53jvb8h5hz83rm0000gn/T/ipykernel_43835/479243729.py:36: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  clo

=== Overall Strategy Leaderboard ===


,Strategy,Win Rate,Avg Return,# Trades,Hold Days
6,Alpha + Minervini,49.2%,5.59%,429,20
2,Minervini (GoldenCross),51.0%,3.39%,575,10
5,Deep Alpha Only (>6.0),44.3%,3.15%,775,10
0,Dip 10%,35.0%,3.14%,1233,10
4,RSI < 30 (Panic),27.6%,1.46%,863,5
3,RSI < 40 (Aggressive),23.0%,0.73%,781,5
1,Dip 15%,30.6%,0.42%,1051,10



=== Detailed Breakdown by Stock (All Strategies) ===


# Trades Avg Return Win Rate
Strategy                Ticker                              
Alpha + Minervini       ANET          13    -10.59%    15.4%
                        ASST          60     -9.01%    33.3%
                        CCJ            9     -4.49%    55.6%
                        CIFR          39      1.25%    41.0%
                        CLS           25      6.97%    56.0%
                        EOSE          39     16.84%    69.2%
                        IREN          37      4.35%    45.9%
                        LAC            9    -17.99%    22.2%
                        LEU           47      3.28%    48.9%
                        NAK           30     17.44%    66.7%
                        NVDA           8     -2.41%    25.0%
                        OKLO          51     10.23%    51.0%
                        QBTS          43     17.72%    60.5%
                        RKLB          19     15.58%    57.9%
Deep Alpha Only (>6.0)  ANET          20     -4.60%    35.0%
                        ASST         130     -1.63%    40.8%
                        CCJ           13      1.12%    46.2%
                        CIFR          51      5.10%    52.9%
                        CLS           25      8.20%    64.0%
                        EOSE          39      5.61%    61.5%
                        INTC          42      3.85%    57.1%
                        IREN          44      5.53%    56.8%
                        LAC           37     -3.35%    40.5%
                        LEU           54      2.27%    46.3%
                        MU            24      2.65%    70.8%
                        NAK           33      3.62%    54.5%
                        NVDA          13      6.30%    84.6%
                        OKLO          57      7.15%    50.9%
                        QBTS          49     12.05%    51.0%
                        RKLB          24      0.88%    50.0%
                        SNDK          16      1.86%     7.5%
Dip 10%                 ANET          20     -4.60%    35.0%
                        ASST         130     -1.63%    40.8%
                        BE            64     11.65%    78.1%
                        CCJ           13      1.12%    46.2%
                        CIFR          51      5.10%    52.9%
                        CLS           25      8.20%    64.0%
                        EOSE          39      5.61%    61.5%
                        FLNC          51     -4.18%    33.3%
                        INTC          42      3.85%    57.1%
                        IREN          44      5.53%    56.8%
                        LAC           37     -3.35%    40.5%
                        LEU           54      2.27%    46.3%
                        MU            24      2.65%    70.8%
                        NAK           33      3.62%    54.5%
                        NLR            5     -5.50%    20.0%
                        NUKZ           7     -2.13%    42.9%
                        NVDA          13      6.30%    84.6%
                        OKLO          57      7.15%    50.9%
                        QBTS          49     12.05%    51.0%
                        RKLB          24      0.88%    50.0%
                        SMAR           0       nan%     0.0%
                        SNDK          16      1.86%     7.5%
                        UNH           20      1.50%    45.0%
                        URA           10     -2.41%    30.0%
                        URNM          10      2.43%    60.0%
Dip 15%                 ANET          15     -3.80%    33.3%
                        ASST         123     -4.88%    34.1%
                        BE            24     -1.87%    41.7%
                        CCJ            3      9.08%   100.0%
                        CIFR          46     -1.98%    47.8%
                        CLS           17      8.25%    58.8%
                        EOSE          30     11.17%    76.7%
                        FLNC          46     -8.59%    28.3%
      